# 06 - Visualisations & Final Report
## London Safety Analysis - Kudzanayi Shepherd Mhlanga

Final notebook. Produces the interactive dashboard, safety map, family recommendation engine, and conclusions.

---

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, str(Path.cwd().parent))
from src.config import DATA_RAW, DATA_PROCESSED, FIGURES_DIR, DASHBOARDS_DIR, BOROUGH_CENTROIDS
from src.scoring import (compute_crime_rates, compute_safety_score,
                          build_borough_summary, TIER_COLOURS)

plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARDS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW / "crime_data_raw.csv", parse_dates=["month"])
df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude","longitude"]).copy()

pop = pd.read_csv(DATA_RAW / "borough_population.csv")
population = dict(zip(pop["borough"], pop["population"]))
summary = build_borough_summary(df, population)
summary["lat"] = summary["borough"].map({b: c[0] for b, c in BOROUGH_CENTROIDS.items()})
summary["lon"] = summary["borough"].map({b: c[1] for b, c in BOROUGH_CENTROIDS.items()})

print(f"Data ready: {len(df):,} records, {df['borough'].nunique()} boroughs")


## 1. Master safety score chart

In [ ]:
tier_colour_map = {
    "Excellent": "#27ae60", "Good": "#2ecc71", "Moderate": "#f1c40f",
    "Elevated": "#e67e22", "High Risk": "#e74c3c"
}

fig, ax = plt.subplots(figsize=(11, 10))
sorted_s = summary.sort_values("safety_score", ascending=True)
colours  = [tier_colour_map.get(str(t), "#95a5a6") for t in sorted_s["safety_tier"]]

bars = ax.barh(sorted_s["borough"], sorted_s["safety_score"],
               color=colours, edgecolor="white", lw=0.6, height=0.75)
for bar, score in zip(bars, sorted_s["safety_score"]):
    ax.text(score + 0.5, bar.get_y() + bar.get_height()/2,
            f"{score:.0f}", va="center", fontsize=8.5, fontweight="bold")

ax.set_xlabel("Safety Score (0 = highest risk, 100 = safest)", fontsize=11)
ax.set_xlim(0, 110)
ax.set_title("London Borough Safety Scores 2024-2025", fontsize=13, fontweight="bold")
ax.spines[["top","right","left"]].set_visible(False)
legend_els = [mpatches.Patch(fc=c, label=t) for t, c in tier_colour_map.items()]
ax.legend(handles=legend_els, loc="lower right", fontsize=9, title="Safety Tier")
plt.figtext(0.01, 0.01, "Source: data.police.uk | Analysis: Kudzanayi Mhlanga",
            fontsize=7, color="#666666")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "safety_scores.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: reports/figures/safety_scores.png")


## 2. Interactive Plotly dashboard

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Safety Score by Borough", "Total Crimes by Borough",
        "Crime Type Breakdown", "Monthly Crime Trend",
    ),
    specs=[[{"type":"bar"},{"type":"bar"}],[{"type":"pie"},{"type":"scatter"}]],
    vertical_spacing=0.18, horizontal_spacing=0.12,
)

s = summary.sort_values("safety_score", ascending=False)
bar_cols = [TIER_COLOURS.get(str(t),"#95a5a6") for t in s["safety_tier"]]
fig.add_trace(go.Bar(x=s["borough"], y=s["safety_score"], marker_color=bar_cols,
    name="Safety Score", showlegend=False,
    hovertemplate="<b>%{x}</b><br>Score: %{y:.0f}<extra></extra>"), row=1, col=1)

s2 = summary.sort_values("total_crimes", ascending=False)
fig.add_trace(go.Bar(x=s2["borough"], y=s2["total_crimes"], marker_color="#e74c3c",
    name="Total Crimes", showlegend=False,
    hovertemplate="<b>%{x}</b><br>Crimes: %{y:,}<extra></extra>"), row=1, col=2)

ct = df["crime_type"].value_counts().head(8)
fig.add_trace(go.Pie(labels=ct.index, values=ct.values, showlegend=True), row=2, col=1)

monthly = df.groupby("month").size().reset_index(name="crimes")
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["crimes"], mode="lines+markers",
    line=dict(color="#3498db", width=2), name="Monthly crimes"), row=2, col=2)

fig.update_layout(
    title_text="<b>London Borough Safety Analysis Dashboard</b>",
    title_x=0.5, height=850, font=dict(family="Arial", size=10),
    plot_bgcolor="white", paper_bgcolor="white",
)
fig.update_xaxes(tickangle=45, tickfont=dict(size=8))

html_path = DASHBOARDS_DIR / "london_safety_dashboard.html"
fig.write_html(str(html_path))
print(f"Dashboard saved: {html_path}")
fig.show()


## 3. Folium safety map

In [ ]:
import folium

tier_folium_colours = {
    "Excellent": "green", "Good": "lightgreen",
    "Moderate": "orange", "Elevated": "red", "High Risk": "darkred",
}

m = folium.Map(location=[51.505, -0.09], zoom_start=10, tiles="CartoDB positron")

for _, row in summary.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue
    tier   = str(row["safety_tier"])
    colour = tier_folium_colours.get(tier, "gray")
    score  = row["safety_score"]
    total  = int(row["total_crimes"])
    
    popup_html = (
        f"<div style=\'font-family:Arial; width:200px\'>"
        f"<h4 style=\'margin:0\'>{row[\'borough\']}</h4><hr>"
        f"<b>Safety Score:</b> {score:.0f}/100<br>"
        f"<b>Tier:</b> {tier}<br>"
        f"<b>Total Crimes:</b> {total:,}"
        f"</div>"
    )
    
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(8, score / 10),
        color=colour, fill=True,
        fill_color=colour, fill_opacity=0.65,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{row[\'borough\']}: {score:.0f}/100 ({tier})"
    ).add_to(m)

map_path = DASHBOARDS_DIR / "london_safety_map.html"
m.save(str(map_path))
print(f"Map saved: {map_path}")


## 4. Family recommendation engine

In [ ]:
house_prices = {
    "Kingston upon Thames": 480000, "Richmond upon Thames": 680000,
    "Sutton": 390000, "Bromley": 440000, "Merton": 490000,
    "Harrow": 420000, "Bexley": 360000, "Havering": 350000,
    "Barnet": 580000, "Hillingdon": 400000, "Ealing": 510000,
    "Hounslow": 430000, "Enfield": 390000, "Redbridge": 420000,
    "Waltham Forest": 460000, "Croydon": 410000, "Lewisham": 460000,
    "Greenwich": 440000, "Brent": 520000, "Barking and Dagenham": 330000,
    "Westminster": 1100000, "Kensington and Chelsea": 1400000,
    "Camden": 820000, "Islington": 720000, "Hackney": 620000,
    "Tower Hamlets": 550000, "Southwark": 560000, "Lambeth": 530000,
    "Wandsworth": 620000, "Hammersmith and Fulham": 750000,
    "Haringey": 580000, "Newham": 400000, "City of London": 950000,
}
outer_london = {
    "Richmond upon Thames","Kingston upon Thames","Bromley","Sutton","Harrow",
    "Bexley","Havering","Barnet","Hillingdon","Enfield","Ealing","Hounslow",
    "Croydon","Redbridge",
}

family_df = summary[["borough","safety_score"]].copy()
family_df["house_price"] = family_df["borough"].map(house_prices)
family_df["is_outer"]    = family_df["borough"].apply(lambda b: 1 if b in outer_london else 0)
family_df = family_df.dropna(subset=["house_price"])

scaler = MinMaxScaler()
family_df["safety_norm"]        = scaler.fit_transform(family_df[["safety_score"]])
family_df["affordability_norm"] = 1 - scaler.fit_transform(family_df[["house_price"]])
family_df["outer_norm"]         = family_df["is_outer"]
family_df["family_score"]       = (
    family_df["safety_norm"]        * 0.50 +
    family_df["affordability_norm"] * 0.30 +
    family_df["outer_norm"]         * 0.20
) * 100
family_df = family_df.sort_values("family_score", ascending=False)

print(f"{'Rank':<5} {'Borough':<35} {'Family':>7} {'Safety':>7} {'Avg Price':>11} {'Zone':<8}")
print("-" * 75)
for rank, (_, row) in enumerate(family_df.head(15).iterrows(), 1):
    zone = "Outer" if row["is_outer"] else "Inner"
    print(f"{rank:<5} {row['borough']:<35} {row['family_score']:>7.1f} "
          f"{row['safety_score']:>7.1f} {int(row['house_price']):>11,} {zone:<8}")


In [ ]:
fig2, ax = plt.subplots(figsize=(11, 8))
top15 = family_df.head(15).sort_values("family_score", ascending=True)
c_fam = ["#27ae60" if r["is_outer"] else "#3498db" for _, r in top15.iterrows()]
bars2 = ax.barh(top15["borough"], top15["family_score"], color=c_fam, edgecolor="white", lw=0.5)
for bar, score in zip(bars2, top15["family_score"]):
    ax.text(score + 0.3, bar.get_y() + bar.get_height()/2,
            f"{score:.1f}", va="center", fontsize=9, fontweight="bold")

ax.legend(handles=[
    mpatches.Patch(fc="#27ae60", label="Outer London"),
    mpatches.Patch(fc="#3498db", label="Inner London"),
], loc="lower right")
ax.set_xlabel("Family Suitability Score (Safety 50% + Affordability 30% + Outer 20%)", fontsize=10)
ax.set_title("Top 15 London Boroughs for Family Relocation", fontsize=13, fontweight="bold")
ax.set_xlim(0, 105)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_family_scores.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Final conclusions

In [ ]:
print("=" * 65)
print("  FINAL REPORT: SAFE AREAS TO LIVE IN LONDON")
print("  London Safety Analysis | Kudzanayi Shepherd Mhlanga")
print("=" * 65)
print()
safest = summary.nlargest(5, "safety_score")
print("SAFEST BOROUGHS:")
for i, (_, row) in enumerate(safest.iterrows(), 1):
    print(f"  {i}. {row['borough']:<35} Score: {row['safety_score']:.0f}/100 [{row['safety_tier']}]")
print()
print("TOP 5 FOR FAMILIES (safety + affordability + location):")
for i, (_, row) in enumerate(family_df.head(5).iterrows(), 1):
    zone = "Outer" if row["is_outer"] else "Inner"
    print(f"  {i}. {row['borough']:<35} Family score: {row['family_score']:.0f} | GBP {int(row['house_price']):,} avg | {zone}")
print()
print("KEY INSIGHTS:")
print("  - ANOVA confirms borough differences are highly significant (p<0.001)")
print("  - Outer London is 40-60% safer on average than central areas")
print("  - Violence and robbery are strongly correlated (r>0.85)")
print("  - Seasonal patterns matter: violence up in summer, burglary up in winter")
print("  - 4 distinct borough crime profiles identified via K-Means clustering")
print()
print("FILES PRODUCED:")
import os
for d in [FIGURES_DIR, DASHBOARDS_DIR, DATA_PROCESSED]:
    for f in sorted(Path(d).glob("*")):
        if f.suffix in [".png",".html",".csv"]:
            size = os.path.getsize(f) / 1024
            print(f"  {str(f.relative_to(Path.cwd().parent))}  ({size:.0f} KB)")
print()
print("Analysis complete. Portfolio project ready.")
